In [1]:
import pandas as pd
from sklearn.metrics import recall_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight

from spam_shield import config
from spam_shield.evaluate import compute_metrics
from spam_shield.prepare import load_split, split_train_test
from spam_shield.train import build_model

train = load_split(config.TRAIN_FILE)
fit_part, validation = split_train_test(train, config.TEST_SIZE, config.RANDOM_STATE)
print(f"Fit part: {len(fit_part):,}   Validation: {len(validation):,}")

Fit part: 24,739   Validation: 6,185


In [3]:
# name: (classifier, use_balanced_weights)
experiments = {
    "A: MultinomialNB (4.1)": (MultinomialNB(), False),
    "B: MultinomialNB + balanced weights": (MultinomialNB(), True),
    "C: ComplementNB": (ComplementNB(), False),
    "D: MultinomialNB alpha=0.1": (MultinomialNB(alpha=0.1), False),
    "E: MultinomialNB + balanced weights + alpha=0.1": (MultinomialNB(alpha=0.1), True),
}

rows = []
for name, (classifier, balanced) in experiments.items():
    model = build_model(classifier)
    if balanced:
        weights = compute_sample_weight("balanced", fit_part["label"])
        model.fit(fit_part, fit_part["label"], classifier__sample_weight=weights)
    else:
        model.fit(fit_part, fit_part["label"])

    predictions = model.predict(validation)
    scores = compute_metrics(validation["label"], predictions)
    per_label = recall_score(validation["label"], predictions, labels=config.LABELS, average=None)
    for label, recall in zip(config.LABELS, per_label):
        scores[f"recall_{label}"] = recall
    rows.append(pd.Series(scores, name=name))
    print("done:", name)

results = pd.DataFrame(rows).round(3)
results

done: A: MultinomialNB (4.1)
done: B: MultinomialNB + balanced weights
done: C: ComplementNB
done: D: MultinomialNB alpha=0.1
done: E: MultinomialNB + balanced weights + alpha=0.1


,accuracy,macro_f1,phishing_recall,recall_ham,recall_spam,recall_phishing,recall_toxic
A: MultinomialNB (4.1),0.848,0.625,0.747,0.988,0.019,0.747,0.585
B: MultinomialNB + balanced weights,0.840,0.798,0.961,0.772,0.922,0.961,0.935
C: ComplementNB,0.896,0.866,0.912,0.911,0.699,0.912,0.856
D: MultinomialNB alpha=0.1,0.893,0.847,0.890,0.904,0.573,0.890,0.887
E: MultinomialNB + balanced weights + alpha=0.1,0.830,0.811,0.943,0.753,0.913,0.943,0.957
